# Requirement 17: Interpreter functionality

> Support file upload, listing, retrieval, deletion, Python execution, execution status, stdout/stderr, generated-file discovery, and health checks.

**Covered by:** Both

**How it is fulfilled:** The sandbox API provides file write/read/list/delete and command execution with exit code, stdout, and stderr. The orchestrator wraps these as the interpreter tool.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()
# Upload
sandbox.write_file("/work/input.csv", "name,amount\nA,1200\nB,800\n")
# List
listing = sandbox.list_files("/work")
print(listing)
check("upload and list", "input.csv" in str(listing))

DirListing(path='/work', entries=[FileInfo(name='input.csv', path='/work/input.csv', size=25, is_directory=False, modified_at=None, mode=420)])
PASS  upload and list


In [3]:
# Python execution with exit code, stdout, and stderr
run = sandbox.exec("cd /work && python -c \"import csv; rows=list(csv.DictReader(open('input.csv'))); print(sum(int(r['amount']) for r in rows)); open('report.txt','w').write('total done')\"")
print("exit code:", run.exit_code, "| stdout:", run.stdout.strip(), "| stderr:", repr(run.stderr))
check("python runs and returns stdout", run.exit_code == 0 and run.stdout.strip() == "2000")
bad = sandbox.exec("python -c 'raise SystemExit(\"boom\")'")
check("failure surfaces exit code and stderr", bad.exit_code != 0 and "boom" in bad.stderr)

exit code:

 0 | stdout: 2000 | stderr: ''
PASS  python runs and returns stdout
PASS  failure surfaces exit code and stderr


In [4]:
# Generated-file discovery, retrieval, deletion
check("generated file is discoverable", "report.txt" in str(sandbox.list_files("/work")))
check("generated file can be retrieved", sandbox.read_file("/work/report.txt") == b"total done")
sandbox.delete_file("/work/input.csv")
check("file deletion works", "input.csv" not in str(sandbox.list_files("/work")))

PASS  generated file is discoverable
PASS  generated file can be retrieved


PASS  file deletion works


In [5]:
# Health and status
info = sandbox.get()
print("state:", info.state, "| stats:", sandbox.get_stats())
check("health check: sandbox is Running", info.state == "Running")
cleanup(sandbox)

state: Running | stats: SandboxStats(cpu=CpuUsage(usage_nano_cores=None, usage_core_nano_seconds=None), memory=ResourceUsage(used_bytes=273567744, total_bytes=2326118400, available_bytes=2052550656), storage=None, network=NetworkUsage(rx_bytes=0, tx_bytes=0))
PASS  health check: sandbox is Running


**Caveat:** Execution status is polled with `exec` and `get`; there is no long-running job object, so asynchronous jobs need the orchestrator to track them.